Step 1 — Import the data

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("fact_shipments.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 60000
Columns: 42


,Shipment_ID,Order_Date,Order_Date_Key,Origin_ID,Destination_ID,Carrier_ID,Supplier_ID,Vehicle_ID,Shipment_Type,Priority,...,Administrative_Cost_USD,Carrier_Payout_USD,Total_Cost_USD,Revenue_USD,Gross_Profit_USD,Gross_Margin_Pct,Estimated_Lost_Revenue_USD,Lead_Time_Days,Cost_per_KM_USD,Cost_per_Shipment_USD
0,SH000001,2025-12-24,20251224,L004,L003,C001,S019,VH0076,Vehicle Delivery,Standard,...,23.57,240.79,299.84,451.75,151.91,33.63,0.0,0.87,2.0750,299.84
1,SH000002,2024-06-10,20240610,L027,L028,C005,S010,VH0118,Vehicle Delivery,Standard,...,9.58,1077.93,1246.71,1488.87,242.16,16.26,0.0,1.39,1.0844,1246.71
2,SH000003,2024-10-04,20241004,L011,L013,C009,S022,VH0136,Vehicle Delivery,Expedited,...,9.71,479.17,544.49,737.18,192.70,26.14,0.0,1.18,1.6826,544.49
3,SH000004,2025-11-06,20251106,L011,L004,C017,S019,VH0099,Freight,Standard,...,22.34,1955.46,2104.12,2525.19,421.06,16.67,0.0,1.66,1.0179,2104.12
4,SH000005,2024-08-30,20240830,L018,L019,C002,S012,VH0115,Parts & Materials,Standard,...,11.66,320.64,422.19,605.77,183.57,30.30,0.0,1.05,1.7446,422.19


Step 2 — Check the data types and missing values

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 42 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Shipment_ID                 60000 non-null  object 
 1   Order_Date                  60000 non-null  object 
 2   Order_Date_Key              60000 non-null  int64  
 3   Origin_ID                   60000 non-null  object 
 4   Destination_ID              60000 non-null  object 
 5   Carrier_ID                  60000 non-null  object 
 6   Supplier_ID                 60000 non-null  object 
 7   Vehicle_ID                  60000 non-null  object 
 8   Shipment_Type               60000 non-null  object 
 9   Priority                    60000 non-null  object 
 10  Shipment_Weight_KG          60000 non-null  float64
 11  Distance_KM                 60000 non-null  float64
 12  Booking_Time                60000 non-null  object 
 13  Assignment_Time             600

In [4]:
df.isnull().sum().sort_values(ascending=False).head(15)

Shipment_ID                  0
Warehouse_Cost_USD           0
SLA_Flag                     0
Damage_Flag                  0
Documentation_Defect_Flag    0
Defect_Flag                  0
Delay_Category               0
Shipment_Status              0
Fuel_Cost_USD                0
Toll_Cost_USD                0
Administrative_Cost_USD      0
Order_Date                   0
Carrier_Payout_USD           0
Total_Cost_USD               0
Revenue_USD                  0
dtype: int64

Step 3 — Check for duplicate records

In [5]:
df.duplicated().sum()

0

In [6]:
df["Shipment_ID"].duplicated().sum()

0

Step 4 — Standardize the date columns

In [7]:
date_columns = [
    "Order_Date",
    "Booking_Time",
    "Assignment_Time",
    "Pickup_Time",
    "Expected_Delivery_Time",
    "Actual_Delivery_Time"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df[date_columns].dtypes

Order_Date                datetime64[ns]
Booking_Time              datetime64[ns]
Assignment_Time           datetime64[ns]
Pickup_Time               datetime64[ns]
Expected_Delivery_Time    datetime64[ns]
Actual_Delivery_Time      datetime64[ns]
dtype: object

In [8]:
df[date_columns].isnull().sum()

Order_Date                0
Booking_Time              0
Assignment_Time           0
Pickup_Time               0
Expected_Delivery_Time    0
Actual_Delivery_Time      0
dtype: int64

Step 5 — Check numeric data quality

In [9]:
numeric_columns = [
    "Shipment_Weight_KG",
    "Distance_KM",
    "Matching_Time_Min",
    "Total_Delay_Hours",
    "Fuel_Cost_USD",
    "Toll_Cost_USD",
    "Warehouse_Cost_USD",
    "Administrative_Cost_USD",
    "Carrier_Payout_USD",
    "Total_Cost_USD",
    "Revenue_USD",
    "Gross_Profit_USD",
    "Lead_Time_Days",
    "Cost_per_KM_USD",
    "Cost_per_Shipment_USD"
]

(df[numeric_columns] < 0).sum().sort_values(ascending=False)

Shipment_Weight_KG         0
Distance_KM                0
Matching_Time_Min          0
Total_Delay_Hours          0
Fuel_Cost_USD              0
Toll_Cost_USD              0
Warehouse_Cost_USD         0
Administrative_Cost_USD    0
Carrier_Payout_USD         0
Total_Cost_USD             0
Revenue_USD                0
Gross_Profit_USD           0
Lead_Time_Days             0
Cost_per_KM_USD            0
Cost_per_Shipment_USD      0
dtype: int64

Step 6 — Business-rule validation

In [10]:
profit_errors = (
    (df["Gross_Profit_USD"] - 
     (df["Revenue_USD"] - df["Total_Cost_USD"])).abs() > 0.01
).sum()

margin_errors = (
    (df["Gross_Margin_Pct"] -
     ((df["Revenue_USD"] - df["Total_Cost_USD"]) /
      df["Revenue_USD"] * 100)).abs() > 0.01
).sum()

print("Profit calculation errors:", profit_errors)
print("Margin calculation errors:", margin_errors)

Profit calculation errors: 6429
Margin calculation errors: 0


Fixing the validation test

In [11]:
profit_diff = (
    df["Gross_Profit_USD"] -
    (df["Revenue_USD"] - df["Total_Cost_USD"])
).abs()

print("Maximum difference:", profit_diff.max())
print("Average difference:", profit_diff.mean())
print("Rows > $0.01:", (profit_diff > 0.01).sum())

Maximum difference: 0.0100000000009004
Average difference: 0.0024931666667264976
Rows > $0.01: 6429


In [12]:
df.loc[profit_diff > 0.01, [
    "Shipment_ID",
    "Revenue_USD",
    "Total_Cost_USD",
    "Gross_Profit_USD"
]].head(10)

,Shipment_ID,Revenue_USD,Total_Cost_USD,Gross_Profit_USD
2,SH000003,737.18,544.49,192.70
3,SH000004,2525.19,2104.12,421.06
27,SH000028,833.25,562.68,270.56
44,SH000045,610.87,405.89,204.97
54,SH000055,2277.92,1680.85,597.06
58,SH000059,765.41,576.64,188.78
70,SH000071,776.49,639.67,136.81
81,SH000082,350.37,262.59,87.77
94,SH000095,1491.04,1081.13,409.92
97,SH000098,1709.49,1335.63,373.87


In [13]:
profit_errors = (
    (df["Gross_Profit_USD"] -
     (df["Revenue_USD"] - df["Total_Cost_USD"])).abs() > 0.011
).sum()

print("Profit calculation errors:", profit_errors)

Profit calculation errors: 0


Step 7 — Exploratory Analysis: Shipment Overview

In [14]:
print("Total shipments:", len(df))
print("Total revenue: $", round(df["Revenue_USD"].sum(), 2))
print("Total logistics spend: $", round(df["Total_Cost_USD"].sum(), 2))
print("Gross profit: $", round(df["Gross_Profit_USD"].sum(), 2))
print("Average OTD: ", round(df["On_Time_Flag"].mean() * 100, 2), "%")
print("Average SLA: ", round(df["SLA_Flag"].mean() * 100, 2), "%")
print("Average lead time:", round(df["Lead_Time_Days"].mean(), 2), "days")

Total shipments: 60000
Total revenue: $ 100073523.43
Total logistics spend: $ 76298065.83
Gross profit: $ 23775456.35
Average OTD:  94.66 %
Average SLA:  92.29 %
Average lead time: 1.48 days


Step 8 — Shipment Type Performance

In [15]:
shipment_type_analysis = (
    df.groupby("Shipment_Type")
      .agg(
          shipments=("Shipment_ID", "count"),
          otd_pct=("On_Time_Flag", "mean"),
          avg_lead_days=("Lead_Time_Days", "mean"),
          avg_cost=("Total_Cost_USD", "mean"),
          avg_margin=("Gross_Margin_Pct", "mean"),
          gross_profit=("Gross_Profit_USD", "sum")
      )
      .reset_index()
)

shipment_type_analysis["otd_pct"] = shipment_type_analysis["otd_pct"] * 100

shipment_type_analysis.round(2)

,Shipment_Type,shipments,otd_pct,avg_lead_days,avg_cost,avg_margin,gross_profit
0,Freight,18046,94.92,1.48,1250.09,25.03,7032115.67
1,Parts & Materials,10663,94.37,1.48,1247.17,25.25,4180019.22
2,Vehicle Delivery,25321,94.63,1.48,1291.26,24.85,10145687.81
3,Warehouse Transfer,5970,94.49,1.49,1297.23,24.81,2417633.65


Step 9 — Priority Performance

In [16]:
priority_analysis = (
    df.groupby("Priority")
      .agg(
          shipments=("Shipment_ID", "count"),
          otd_pct=("On_Time_Flag", "mean"),
          avg_delay_hours=("Total_Delay_Hours", "mean"),
          avg_lead_days=("Lead_Time_Days", "mean"),
          avg_cost=("Total_Cost_USD", "mean"),
          avg_margin=("Gross_Margin_Pct", "mean"),
          gross_profit=("Gross_Profit_USD", "sum")
      )
      .reset_index()
)

priority_analysis["otd_pct"] = priority_analysis["otd_pct"] * 100

priority_analysis.round(2)

,Priority,shipments,otd_pct,avg_delay_hours,avg_lead_days,avg_cost,avg_margin,gross_profit
0,Critical,4745,94.46,4.53,1.47,1269.36,24.96,1863082.80
1,Expedited,14336,94.71,4.57,1.48,1276.36,24.98,5693612.45
2,Standard,40919,94.66,4.57,1.48,1270.24,24.97,16218761.10


Step 10 — Delay Category Analysis in Python

In [17]:
delay_analysis = (
    df.groupby("Delay_Category")
      .agg(
          shipments=("Shipment_ID", "count"),
          total_delay_hours=("Total_Delay_Hours", "sum"),
          avg_delay_hours=("Total_Delay_Hours", "mean"),
          avg_otd=("On_Time_Flag", "mean"),
          lost_revenue=("Estimated_Lost_Revenue_USD", "sum")
      )
      .reset_index()
)

delay_analysis["avg_otd"] = delay_analysis["avg_otd"] * 100

delay_analysis.round(2)

,Delay_Category,shipments,total_delay_hours,avg_delay_hours,avg_otd,lost_revenue
0,Pickup,20552,91112.84,4.43,96.51,20611.04
1,Transit,28455,139914.83,4.92,93.15,57777.72
2,Unloading,10993,42892.92,3.90,95.10,13162.61


Step 11 — Carrier Performance in Python

In [18]:
carrier_analysis = (
    df.groupby("Carrier_ID")
      .agg(
          shipments=("Shipment_ID", "count"),
          otd_pct=("On_Time_Flag", "mean"),
          avg_delay_hours=("Total_Delay_Hours", "mean"),
          avg_margin=("Gross_Margin_Pct", "mean"),
          avg_carrier_cost=("Carrier_Payout_USD", "mean"),
          gross_profit=("Gross_Profit_USD", "sum")
      )
      .reset_index()
)

carrier_analysis["otd_pct"] = carrier_analysis["otd_pct"] * 100

carrier_analysis.sort_values(
    ["otd_pct", "avg_margin"],
    ascending=[True, True]
).round(2).head(10)

,Carrier_ID,shipments,otd_pct,avg_delay_hours,avg_margin,avg_carrier_cost,gross_profit
24,C025,1995,92.68,5.75,24.84,1150.63,791955.20
26,C027,2040,92.94,5.55,24.97,1129.44,804191.64
9,C010,1976,93.12,5.59,25.10,1121.85,790892.06
19,C020,1996,93.34,4.85,25.10,1115.34,790459.36
6,C007,2037,93.57,5.31,24.74,1123.16,797272.45
21,C022,2021,93.67,5.60,24.92,1093.91,776138.15
8,C009,2030,93.89,4.65,25.01,1166.59,836636.56
3,C004,2031,93.94,4.83,24.94,1127.75,812774.52
1,C002,2020,93.96,4.75,24.79,1123.87,786689.11
4,C005,2050,94.05,4.61,24.97,1140.85,822220.79


Step 12 — Supplier Performance in Python

In [19]:
supplier_analysis = (
    df.groupby("Supplier_ID")
      .agg(
          shipments=("Shipment_ID", "count"),
          sla_pct=("SLA_Flag", "mean"),
          defect_rate=("Defect_Flag", "mean"),
          avg_lead_days=("Lead_Time_Days", "mean"),
          avg_cost=("Total_Cost_USD", "mean"),
          gross_profit=("Gross_Profit_USD", "sum")
      )
      .reset_index()
)

supplier_analysis["sla_pct"] = supplier_analysis["sla_pct"] * 100
supplier_analysis["defect_rate"] = supplier_analysis["defect_rate"] * 100

supplier_analysis.sort_values(
    ["sla_pct", "defect_rate"],
    ascending=[True, False]
).round(2).head(10)

,Supplier_ID,shipments,sla_pct,defect_rate,avg_lead_days,avg_cost,gross_profit
32,S033,1444,90.51,2.77,1.51,1266.07,573459.52
34,S035,1500,90.87,4.27,1.49,1259.70,594521.05
12,S013,1467,91.14,3.75,1.52,1309.17,600178.53
29,S030,1460,91.16,2.74,1.49,1266.95,579610.46
35,S036,1518,91.30,2.83,1.51,1309.86,622488.09
39,S040,1492,91.49,3.62,1.49,1264.23,590177.36
26,S027,1500,91.53,3.00,1.47,1277.61,603937.22
9,S010,1548,91.60,3.81,1.46,1233.35,591232.25
33,S034,1525,91.61,2.82,1.48,1268.48,602554.00
0,S001,1460,91.71,3.97,1.51,1291.19,584858.67


Step 13 — Route Performance in Python

In [20]:
route_analysis = (
    df.groupby(["Origin_ID", "Destination_ID"])
      .agg(
          shipments=("Shipment_ID", "count"),
          otd_pct=("On_Time_Flag", "mean"),
          avg_delay_hours=("Total_Delay_Hours", "mean"),
          avg_distance_km=("Distance_KM", "mean"),
          avg_margin=("Gross_Margin_Pct", "mean"),
          gross_profit=("Gross_Profit_USD", "sum")
      )
      .reset_index()
)

route_analysis["otd_pct"] = route_analysis["otd_pct"] * 100

route_analysis = route_analysis[
    route_analysis["shipments"] >= 50
]

route_analysis.sort_values(
    ["otd_pct", "avg_margin"],
    ascending=[True, True]
).round(2).head(10)

,Origin_ID,Destination_ID,shipments,otd_pct,avg_delay_hours,avg_distance_km,avg_margin,gross_profit
717,L025,L022,319,84.95,4.52,2088.8,22.38,203322.13
632,L022,L025,332,87.05,4.55,2088.8,22.13,210891.96
633,L022,L026,287,89.55,4.81,1829.8,23.04,170899.69
603,L021,L025,288,89.93,4.73,1885.4,21.92,163616.34
716,L025,L021,333,90.39,4.51,1885.4,22.70,198763.80
746,L026,L022,322,90.68,4.64,1829.8,23.00,190715.20
745,L026,L021,286,91.61,4.94,1621.7,22.76,148862.60
661,L023,L025,291,91.75,4.77,1333.8,23.41,131587.18
266,L010,L006,347,92.51,4.62,1604.8,22.73,178095.39
153,L006,L010,272,93.01,4.55,1604.8,22.61,140874.15


Step 14 — Cost Structure Analysis

In [21]:
cost_analysis = pd.DataFrame({
    "Cost_Category": [
        "Fuel",
        "Tolls",
        "Warehouse",
        "Administrative",
        "Carrier Payout"
    ],
    "Total_Cost": [
        df["Fuel_Cost_USD"].sum(),
        df["Toll_Cost_USD"].sum(),
        df["Warehouse_Cost_USD"].sum(),
        df["Administrative_Cost_USD"].sum(),
        df["Carrier_Payout_USD"].sum()
    ]
})

cost_analysis["Share_pct"] = (
    cost_analysis["Total_Cost"] /
    cost_analysis["Total_Cost"].sum() * 100
)

cost_analysis.sort_values(
    "Total_Cost",
    ascending=False
).round(2)

,Cost_Category,Total_Cost,Share_pct
4,Carrier Payout,67464146.81,88.42
1,Tolls,4108519.80,5.38
2,Warehouse,3207107.60,4.20
3,Administrative,959029.69,1.26
0,Fuel,559264.82,0.73


Step 15 — Monthly Financial Trend

In [22]:
monthly_analysis = (
    df.assign(Month=df["Order_Date"].dt.to_period("M").astype(str))
      .groupby("Month")
      .agg(
          shipments=("Shipment_ID", "count"),
          revenue=("Revenue_USD", "sum"),
          logistics_spend=("Total_Cost_USD", "sum"),
          gross_profit=("Gross_Profit_USD", "sum"),
          avg_margin=("Gross_Margin_Pct", "mean"),
          otd_pct=("On_Time_Flag", "mean"),
          avg_cost_per_shipment=("Cost_per_Shipment_USD", "mean"),
          avg_cost_per_km=("Cost_per_KM_USD", "mean")
      )
      .reset_index()
)

monthly_analysis["otd_pct"] = monthly_analysis["otd_pct"] * 100

monthly_analysis.round(2).head(10)

,Month,shipments,revenue,logistics_spend,gross_profit,avg_margin,otd_pct,avg_cost_per_shipment,avg_cost_per_km
0,2024-01,2584,4253205.79,3254041.62,999164.31,24.81,94.23,1259.30,1.30
1,2024-02,2383,3972104.26,3024448.13,947656.25,25.11,94.38,1269.18,1.30
2,2024-03,2590,4145417.38,3161625.31,983791.94,25.08,94.71,1220.70,1.32
3,2024-04,2411,4114474.83,3141640.67,972834.14,24.82,94.07,1303.04,1.29
4,2024-05,2461,4157976.84,3173240.08,984736.93,24.99,94.31,1289.41,1.30
5,2024-06,2408,4011972.19,3057586.93,954385.57,25.07,94.02,1269.76,1.30
6,2024-07,2569,4369961.40,3342705.90,1027255.05,24.76,94.51,1301.17,1.29
7,2024-08,2606,4243516.87,3236792.16,1006724.97,24.94,94.78,1242.05,1.31
8,2024-09,2477,4149319.79,3163378.32,985941.62,24.97,94.67,1277.10,1.29
9,2024-10,2544,4195504.87,3196681.20,998823.84,25.09,94.34,1256.56,1.30


Step 16 — Identify Cost Outliers

In [23]:
Q1 = df["Cost_per_KM_USD"].quantile(0.25)
Q3 = df["Cost_per_KM_USD"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

cost_outliers = df[
    (df["Cost_per_KM_USD"] < lower_bound) |
    (df["Cost_per_KM_USD"] > upper_bound)
].copy()

print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Cost outliers:", len(cost_outliers))
print("Outlier percentage:", round(len(cost_outliers) / len(df) * 100, 2), "%")

Lower bound: 0.57
Upper bound: 1.9
Cost outliers: 3702
Outlier percentage: 6.17 %


In [24]:
cost_outliers[
    [
        "Shipment_ID",
        "Distance_KM",
        "Total_Cost_USD",
        "Cost_per_KM_USD",
        "Revenue_USD",
        "Gross_Margin_Pct"
    ]
].sort_values("Cost_per_KM_USD", ascending=False).head(10)

,Shipment_ID,Distance_KM,Total_Cost_USD,Cost_per_KM_USD,Revenue_USD,Gross_Margin_Pct
20114,SH020115,68.6,415.07,6.0506,651.00,36.24
39932,SH039933,68.6,409.71,5.9724,667.53,38.62
56259,SH056260,68.6,396.27,5.7765,531.29,25.41
25469,SH025470,68.6,384.99,5.6121,502.36,23.36
34766,SH034767,68.6,379.46,5.5315,523.63,27.53
50087,SH050088,68.6,372.89,5.4357,504.24,26.05
43452,SH043453,68.6,370.68,5.4035,549.13,32.50
14991,SH014992,68.6,370.67,5.4034,518.95,28.57
38051,SH038052,68.6,368.23,5.3678,486.54,24.32
4469,SH004470,68.6,367.63,5.3590,504.87,27.18


Step 17 — Profitability Outliers

In [25]:
Q1 = df["Gross_Margin_Pct"].quantile(0.25)
Q3 = df["Gross_Margin_Pct"].quantile(0.75)

IQR = Q3 - Q1

lower_margin_bound = Q1 - 1.5 * IQR

margin_outliers = df[
    df["Gross_Margin_Pct"] < lower_margin_bound
].copy()

print("Lower margin bound:", round(lower_margin_bound, 2), "%")
print("Low-margin shipments:", len(margin_outliers))
print(
    "Low-margin percentage:",
    round(len(margin_outliers) / len(df) * 100, 2),
    "%"
)

Lower margin bound: 7.49 %
Low-margin shipments: 0
Low-margin percentage: 0.0 %


In [26]:
margin_outliers[
    [
        "Shipment_ID",
        "Revenue_USD",
        "Total_Cost_USD",
        "Gross_Profit_USD",
        "Gross_Margin_Pct",
        "Distance_KM",
        "Cost_per_KM_USD"
    ]
].sort_values("Gross_Margin_Pct").head(10)

,Shipment_ID,Revenue_USD,Total_Cost_USD,Gross_Profit_USD,Gross_Margin_Pct,Distance_KM,Cost_per_KM_USD


Step 18 — Identify the Most Expensive Routes

In [27]:
route_cost = (
    df.groupby(["Origin_ID", "Destination_ID"])
      .agg(
          shipments=("Shipment_ID", "count"),
          avg_distance_km=("Distance_KM", "mean"),
          avg_cost_per_km=("Cost_per_KM_USD", "mean"),
          avg_cost_per_shipment=("Cost_per_Shipment_USD", "mean"),
          avg_margin=("Gross_Margin_Pct", "mean"),
          gross_profit=("Gross_Profit_USD", "sum")
      )
      .reset_index()
)

route_cost = route_cost[
    route_cost["shipments"] >= 50
]

route_cost.sort_values(
    "avg_cost_per_km",
    ascending=False
).round(2).head(10)

,Origin_ID,Destination_ID,shipments,avg_distance_km,avg_cost_per_km,avg_cost_per_shipment,avg_margin,gross_profit
540,L019,L020,378,68.6,3.39,232.64,32.35,42326.02
569,L020,L019,374,68.6,3.37,230.95,33.05,42615.70
60,L003,L004,505,144.5,2.11,305.39,30.38,68193.94
89,L004,L003,530,144.5,2.11,304.48,29.99,69938.51
660,L023,L024,278,147.3,2.10,309.37,30.42,37914.28
689,L024,L023,316,147.3,2.09,308.26,30.83,43827.35
508,L018,L016,381,158.1,2.03,320.36,30.23,53345.58
451,L016,L018,364,158.1,1.98,313.61,30.30,50160.51
511,L018,L020,398,175.9,1.91,336.01,29.54,56825.35
568,L020,L018,382,175.9,1.91,335.93,29.64,54828.18


Step 19 — Vehicle Utilization Analysis

In [28]:
vehicle_analysis = (
    df.groupby("Vehicle_ID")
      .agg(
          shipments=("Shipment_ID", "count"),
          total_distance_km=("Distance_KM", "sum"),
          avg_distance_km=("Distance_KM", "mean"),
          avg_cost_per_km=("Cost_per_KM_USD", "mean"),
          avg_otd=("On_Time_Flag", "mean"),
          gross_profit=("Gross_Profit_USD", "sum")
      )
      .reset_index()
)

vehicle_analysis["otd_pct"] = vehicle_analysis["avg_otd"] * 100

vehicle_analysis.sort_values(
    "shipments",
    ascending=False
).round(2).head(10)

,Vehicle_ID,shipments,total_distance_km,avg_distance_km,avg_cost_per_km,avg_otd,gross_profit,otd_pct
65,VH0066,452,503677.5,1114.33,1.28,0.95,174370.14,94.91
135,VH0136,443,553097.8,1248.53,1.27,0.92,188219.91,92.33
67,VH0068,440,473564.6,1076.28,1.30,0.95,166479.84,95.45
36,VH0037,439,480467.3,1094.46,1.27,0.96,173305.46,96.36
80,VH0081,439,490721.3,1117.82,1.30,0.95,166969.72,95.44
19,VH0020,439,502134.3,1143.81,1.29,0.94,175720.62,94.08
45,VH0046,436,519012.3,1190.40,1.29,0.95,179345.94,95.41
91,VH0092,434,532136.7,1226.12,1.29,0.95,180849.12,94.93
39,VH0040,434,520170.4,1198.55,1.29,0.95,179737.16,94.93
66,VH0067,433,495792.6,1145.02,1.34,0.94,166341.62,94.00


Step 20 — Correlation Analysis

In [29]:
correlation_columns = [
    "Distance_KM",
    "Matching_Time_Min",
    "Total_Delay_Hours",
    "Lead_Time_Days",
    "Total_Cost_USD",
    "Revenue_USD",
    "Gross_Profit_USD",
    "Gross_Margin_Pct",
    "Cost_per_KM_USD"
]

correlation_matrix = df[correlation_columns].corr().round(2)

correlation_matrix

,Distance_KM,Matching_Time_Min,Total_Delay_Hours,Lead_Time_Days,Total_Cost_USD,Revenue_USD,Gross_Profit_USD,Gross_Margin_Pct,Cost_per_KM_USD
Distance_KM,1.00,-0.00,0.00,0.95,0.99,0.98,0.88,-0.33,-0.49
Matching_Time_Min,-0.00,1.00,-0.00,0.02,-0.00,-0.00,0.00,-0.00,0.00
Total_Delay_Hours,0.00,-0.00,1.00,0.14,0.01,0.00,0.00,-0.00,0.00
Lead_Time_Days,0.95,0.02,0.14,1.00,0.93,0.93,0.83,-0.31,-0.46
Total_Cost_USD,0.99,-0.00,0.01,0.93,1.00,0.99,0.89,-0.33,-0.44
Revenue_USD,0.98,-0.00,0.00,0.93,0.99,1.00,0.94,-0.25,-0.44
Gross_Profit_USD,0.88,0.00,0.00,0.83,0.89,0.94,1.00,0.02,-0.40
Gross_Margin_Pct,-0.33,-0.00,-0.00,-0.31,-0.33,-0.25,0.02,1.00,0.29
Cost_per_KM_USD,-0.49,0.00,0.00,-0.46,-0.44,-0.44,-0.40,0.29,1.00


In [30]:
profit_correlation = (
    correlation_matrix["Gross_Profit_USD"]
    .drop("Gross_Profit_USD")
    .sort_values(ascending=False)
)

profit_correlation

Revenue_USD          0.94
Total_Cost_USD       0.89
Distance_KM          0.88
Lead_Time_Days       0.83
Gross_Margin_Pct     0.02
Matching_Time_Min    0.00
Total_Delay_Hours    0.00
Cost_per_KM_USD     -0.40
Name: Gross_Profit_USD, dtype: float64

Step 21 — Create the final Python-cleaned dataset

In [31]:
df_clean = df.copy()

df_clean.to_csv(
    "fact_shipments_clean.csv",
    index=False
)

print("Cleaned dataset saved successfully.")
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))

Cleaned dataset saved successfully.
Rows: 60000
Columns: 42
